# KV Cache Compression Benchmark Pipeline

This notebook allows you to automatically execute and compare multiple KV cache compression techniques across LongBench and RULER benchmarks.

### "Ours" vs "CommonKV"
In this repository, **"ours"** and **"commonkv"** refer to the exact same methodology (the paper's proposed cross-layer SVD sharing). The codebase uses the flag `commonkv` when patching LLama models and `ours` when patching Mistral models for historical reasons.



In [1]:
!hf auth whoami


user:  ArchaeonSeq
orgs:  ArchSeq,discord-community,mcp-course


In [2]:
import os
import sys
import subprocess
import time
import json
import torch
import pandas as pd
from IPython.display import display, HTML

# Add repo path
sys.path.append(os.path.abspath('.'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')



Using device: cuda


## 1. Define Pipeline Configuration


In [3]:
# MODEL_PATH = "Qwen/Qwen2.5-0.5B-Instruct" # 'meta-llama/Meta-Llama-3.1-8B-Instruct'
# MODEL_PATH = "mistralai/Mistral-7B-Instruct-v0.2"
MODEL_PATH = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

DATASET = 'narrativeqa' # Example LongBench dataset. Options: qasper, multifieldqa_en, hotpotqa, etc.

# The list of all compression methods to evaluate
# Options: 'fullkv', 'commonkv' (llama), 'ours' (mistral), 'think', 'palu', 'minicache', 'snapkv', 'pyramidkv', 'h2o', 'my_custom_method'
METHODS_TO_TEST = ['fullkv', 'commonkv', 'think', 'palu', 'minicache']

# CommonKV / Baseline Parameters
RANK = 4096
LAYER_STEP = 2
MAX_CAPACITY_PROMPTS = 8950



## 2. Execute Benchmark Pipeline
Because the repository relies on **monkeypatching** the HuggingFace `transformers` global namespace to inject the different KV caching techniques, loading multiple methods sequentially in the same Python process will cause collisions and crashes.

To safely and programatically run all of them, we execute the repo's evaluation scripts (`run_longbench.py` and `run_ruler.py`) in isolated subprocesses.

**Note**: To evaluate `my_custom_method` on these benchmarks, you will need to add your custom logic hooks inside `commonkv/monkeypatch.py` so the `run_longbench.py` script can pick it up!



In [4]:
!sed -i 's/from modeling_llama/from commonkv.modeling_llama/g' commonkv/monkeypatch.py
!sed -i 's/from modeling_mistral/from commonkv.modeling_mistral/g' commonkv/monkeypatch.py
!sed -i 's/from llama_model/from commonkv.llama_model/g' commonkv/monkeypatch.py
!sed -i 's/from mistral_model/from commonkv.mistral_model/g' commonkv/monkeypatch.py


In [5]:
# %%bash
# git clone https://github.com/Zefan-Cai/PyramidKV.git /tmp/PyramidKV
# # If the repo contains an inner pyramidkv folder, extract it. 
# # Otherwise, treat the entire repo as the pyramidkv module itself!
# if [ -d "/tmp/PyramidKV/pyramidkv" ]; then 
#     cp -r /tmp/PyramidKV/pyramidkv ./
# else 
#     cp -r /tmp/PyramidKV ./pyramidkv
# fi

In [6]:
# %%bash
# # 1. Force the exact transformers version (>=4.44) the authors wrote patches for 
# #    before Hugging Face removed `SlidingWindowCache` in 4.45+
# pip install "transformers==4.44.2"

# # 2. Install flash-attention (Required by llama_model.py)
# #    Note: Compiling this from source might take ~10-15 minutes on Kaggle!
# pip install flash-attn --no-build-isolation

# # 3. Clone PyramidKV
# git clone https://github.com/Zefan-Cai/PyramidKV.git /tmp/PyramidKV

# # 4. Generate a dynamic setup.py for it! 
# #    Because the authors didn't include one, we'll force Python to recognize 
# #    the root scripts or inner folders natively as a named module.
# cat << 'EOF' > /tmp/PyramidKV/setup.py
# from setuptools import setup, find_packages
# import os

# # Grab all standalone python scripts in the repo to bundle as modules
# modules = [f[:-3] for f in os.listdir('.') if f.endswith('.py') and f != 'setup.py']

# setup(
#     name="pyramidkv",
#     version="0.1.0",
#     packages=find_packages(),
#     py_modules=modules
# )
# EOF

# # 5. Install it as a native python package using our minted setup.py
# pip install -e /tmp/PyramidKV


In [7]:
# import torch
# !pip install flash_attn

In [ ]:
longbench_results_dir = "./results_long_bench"
os.makedirs(longbench_results_dir, exist_ok=True)

for method in METHODS_TO_TEST:
    print(f"\n{'='*50}\nEvaluating Method: {method.upper()}\n{'='*50}")
    
    # We swap 'commonkv' <-> 'ours' dynamically if user mismatched the model type
    eval_method = method
    if 'llama' in MODEL_PATH.lower() and method == 'ours':
        eval_method = 'commonkv'
    if 'mistral' in MODEL_PATH.lower() and method == 'commonkv':
        eval_method = 'ours'

    # # Build the command for LongBench
    # cmd = [
    #     "python3", "run_longbench.py",
    #     "--method", eval_method,
    #     "--model_path", MODEL_PATH,
    #     "--dataset", DATASET,
    #     "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
    #     "--attn_implementation", "sdpa",  # Switch to 'eager' if FA2 fails
    #     "--layer_step", str(LAYER_STEP),
    #     "--rank", str(RANK),
    #     "--save_dir", longbench_results_dir,
    #     "--eval_batch_size", "1"
    # ]


    # Build the command for LongBench
    cmd = [
        "python3", "run_longbench.py",
        "--method", eval_method,
        "--model_path", MODEL_PATH,
        "--dataset", DATASET,
        "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
        "--attn_implementation", "sdpa",
        "--layer_step", str(LAYER_STEP),
        "--rank", str(RANK),
        "--save_dir", longbench_results_dir,
        "--eval_batch_size", "1",
        "--steps", "5",  # <--- Add this flag and target number here!
        "--max_datasets", "2"
    ]

    
    start_time = time.time()
    
    # Run the subprocess
    try:
        # We stream output to notebook seamlessly
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
        process.wait()
        
        if process.returncode == 0:
            print(f"\n✅ Successfully completed {method} in {time.time() - start_time:.1f}s")
        else:
            print(f"\n❌ Failed or halted {method} (Return code: {process.returncode})")
            
    except KeyboardInterrupt:
        print("\nPipeline halted by user.")
        process.kill()
        break




Evaluating Method: FULLKV
Working on max_capacity_prompts 8950 dataset narrativeqa - 1/2
Loading data...
Max Length is 36418
Finish loading model and tokenizer

  0%|          | 0/200 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (35534 > 2048). Running this sequence through the model will result in indexing errors
This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.

  1%|          | 2/200 [00:45<1:17:27, 23.47s/it]


## 3. Compare Results
The `run_longbench.py` script dumps `.json` files containing the model's generated text predictions. 
The official LongBench metrics (like F1 score, Exact Match, etc.) require the `eval.py` script from the official THUDM/LongBench repository. 

However, we can look at the generations and lengths directly generated from our pipeline loop!



In [ ]:
# Find the dynamic subfolder created by run_longbench.py
import glob

# The script creates folders like Llama-3.1-8B-Instruct_4096_2_v4/dataset/
model_basename = MODEL_PATH.split('/')[-1]
search_pattern = f"{longbench_results_dir}/{model_basename}_*_{LAYER_STEP}_*/{DATASET}/*.json"
json_files = glob.glob(search_pattern)

results = []
for file_path in json_files:
    method_name = os.path.basename(file_path).replace('.json', '')
    
    try:
        with open(file_path, 'r') as f:
            # Depending on if it's a JSONL or JSON
            content = f.read().strip().split('\n')
            completed_tasks = len(content)
            
            # Read just the first prediction for a sanity check preview
            if completed_tasks > 0:
                first_entry = json.loads(content[0])
                pred_preview = first_entry.get('pred', '')[:100] + '...'
            else:
                pred_preview = 'N/A'
                
            results.append({
                "Method": method_name,
                "Datapoints Evaluated": completed_tasks,
                "Sample Prediction": pred_preview,
                "Output File": file_path
            })
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

if results:
    df = pd.DataFrame(results)
    display(df)
else:
    print("No result JSON files found. Did the pipeline finish successfully?")



## 4. RULER Benchmark
You can run an identical loop for RULER using `run_ruler.py` simply by swapping the command in Section 2!
